In [ ]:
!pip install -q ultralytics roboflow

import os, time, shutil, pathlib, yaml, psutil
import numpy as np
import torch
import ultralytics
from ultralytics import YOLO

ultralytics.checks()
assert torch.cuda.is_available(), "Runtime sem GPU: Ambiente de execução > Alterar tipo"

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB  = torch.cuda.get_device_properties(0).total_memory / 1e9
IS_A100  = "A100" in GPU_NAME
print(f"{GPU_NAME} | {VRAM_GB:.0f} GB | A100={IS_A100}")

Ultralytics 8.4.130 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Setup complete ✅ (12 CPUs, 83.5 GB RAM, 47.1/235.7 GB disk)
NVIDIA A100-SXM4-40GB | 42 GB | A100=True


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = pathlib.Path("/content/drive/MyDrive/football_analysis/runs")
LOCAL = pathlib.Path("/content/runs")          # treino escreve aqui
DRIVE.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PRESET = "qualidade_maxima"      # rapido | equilibrado | qualidade_maxima
IMGSZ  = {"rapido": 640, "equilibrado": 960, "qualidade_maxima": 1280}[PRESET]

if not IS_A100 and IMGSZ == 1280:
    print(f"AVISO: {GPU_NAME} com {VRAM_GB:.0f} GB. 1280 roda, mas o batch cai "
          f"e a época fica ~3x mais lenta. Considere PRESET='equilibrado'.")

BATCH    = 0.70                  # fração da VRAM (AutoBatch)
RUN_NAME = f"yolo26x_{PRESET}_{IMGSZ}"

In [ ]:
torch.set_float32_matmul_precision("high")   # API atual; cobre o matmul.allow_tf32
torch.backends.cudnn.allow_tf32 = True       # já é True por padrão, mas explícito não dói

WORKERS = min(8, os.cpu_count() or 4)
RAM_GB  = psutil.virtual_memory().available / 1e9
CACHE   = "ram" if RAM_GB > 20 else "disk"
print(f"workers={WORKERS} | RAM livre={RAM_GB:.0f} GB | cache={CACHE}")

workers=8 | RAM livre=87 GB | cache=ram


In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf      = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
dataset = project.version(1).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to football-players-detection-1 in yolov8:: 100%|██████████| 1338/1338 [00:00<00:00, 3728.00it/s]


In [ ]:
loc  = pathlib.Path(dataset.location)
DATA = loc / "data.yaml"
cfg  = yaml.safe_load(DATA.read_text())
cfg.update(path=str(loc), train="train/images", val="valid/images", test="test/images")
DATA.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg["names"])

['ball', 'goalkeeper', 'player', 'referee']


In [ ]:
RUN_DIR = LOCAL / RUN_NAME
LAST    = RUN_DIR / "weights" / "last.pt"
DRIVE_LAST = DRIVE / RUN_NAME / "weights" / "last.pt"

# sessão caiu? traz o checkpoint do Drive de volta
if not LAST.exists() and DRIVE_LAST.exists():
    shutil.copytree(DRIVE / RUN_NAME, RUN_DIR, dirs_exist_ok=True)
    print("checkpoint recuperado do Drive")

def sync_to_drive(trainer):
    shutil.copytree(trainer.save_dir, DRIVE / RUN_NAME, dirs_exist_ok=True)

if LAST.exists():
    model = YOLO(str(LAST))
    model.add_callback("on_model_save", sync_to_drive)
    model.train(resume=True)
else:
    model = YOLO("yolo26x.pt")
    model.add_callback("on_model_save", sync_to_drive)
    model.train(
        data=str(DATA),
        epochs=100, imgsz=IMGSZ, batch=BATCH,
        cache=CACHE, workers=WORKERS, patience=30,
        deterministic=False,
        compile="default",
        channels_last=True,          # Tensor Core gosta de NHWC
        save_period=10,
        project=str(LOCAL), name=RUN_NAME, exist_ok=True,
        plots=True,
    )
sync_to_drive(model.trainer)

Ultralytics 8.4.130 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=0.7, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=True, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=default, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/football-players-detection-1/data.yaml, degrees=0.0, deterministic=False, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name

W0827 16:28:20.668000 6470 torch/utils/_sympy/interp.py:179] [3/1] failed while executing pow_by_natural([VR[2, int_oo], VR[-1, -1]])
W0827 16:28:20.671000 6470 torch/utils/_sympy/interp.py:179] [3/1] failed while executing pow_by_natural([VR[2, int_oo], VR[-1, -1]])
W0827 16:28:20.674000 6470 torch/utils/_sympy/interp.py:179] [3/1] failed while executing pow_by_natural([VR[2, int_oo], VR[-1, -1]])
W0827 16:28:20.678000 6470 torch/utils/_sympy/interp.py:179] [3/1] failed while executing pow_by_natural([VR[2, int_oo], VR[-1, -1]])
W0827 16:28:20.680000 6470 torch/utils/_sympy/interp.py:179] [3/1] failed while executing pow_by_natural([VR[2, int_oo], VR[-1, -1]])
W0827 16:28:20.682000 6470 torch/utils/_sympy/interp.py:179] [3/1] failed while executing pow_by_natural([VR[2, int_oo], VR[-1, -1]])
W0827 16:28:29.546000 6470 torch/utils/_sympy/interp.py:179] [3/2] failed while executing pow_by_natural([VR[2, int_oo], VR[-1, -1]])
W0827 16:28:29.549000 6470 torch/utils/_sympy/interp.py:179] [

    58817808       834.2         7.055      8.61e+04           nan      (1, 3, 1280, 1280)                    list


W0827 16:31:38.685000 6470 torch/utils/_sympy/interp.py:179] [3/4] failed while executing pow_by_natural([VR[2, int_oo], VR[-1, -1]])
W0827 16:31:38.690000 6470 torch/utils/_sympy/interp.py:179] [3/4] failed while executing pow_by_natural([VR[2, int_oo], VR[-1, -1]])
W0827 16:31:38.693000 6470 torch/utils/_sympy/interp.py:179] [3/4] failed while executing pow_by_natural([VR[2, int_oo], VR[-1, -1]])
W0827 16:31:38.696000 6470 torch/utils/_sympy/interp.py:179] [3/4] failed while executing pow_by_natural([VR[2, int_oo], VR[-1, -1]])
W0827 16:31:38.699000 6470 torch/utils/_sympy/interp.py:179] [3/4] failed while executing pow_by_natural([VR[2, int_oo], VR[-1, -1]])
W0827 16:31:38.701000 6470 torch/utils/_sympy/interp.py:179] [3/4] failed while executing pow_by_natural([VR[2, int_oo], VR[-1, -1]])
W0827 16:31:40.652000 6470 torch/utils/_sympy/interp.py:179] [3/5] failed while executing pow_by_natural([VR[2, int_oo], VR[-1, -1]])
W0827 16:31:40.656000 6470 torch/utils/_sympy/interp.py:179] [

    58817808        1668        15.649     2.101e+05           nan      (2, 3, 1280, 1280)                    list
    58817808        3337        29.691         76.51           nan      (4, 3, 1280, 1280)                    list
    58817808        6673        57.271         136.7           nan      (8, 3, 1280, 1280)                    list
CUDA out of memory. Tried to allocate 150.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 85.44 MiB is free. Including non-PyTorch memory, this process has 39.40 GiB memory in use. Of the allocated memory 38.55 GiB is allocated by PyTorch, and 318.22 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
CUDA out of memory. Tried to allocate 300.00 MiB. GPU 0 has a total capa

W0827 16:36:06.162000 6470 torch/_dynamo/convert_frame.py:1743] [3/8] torch._dynamo hit config.recompile_limit (8)
W0827 16:36:06.162000 6470 torch/_dynamo/convert_frame.py:1743] [3/8]    function: 'forward' (/usr/local/lib/python3.13/dist-packages/ultralytics/nn/modules/conv.py:69)
W0827 16:36:06.162000 6470 torch/_dynamo/convert_frame.py:1743] [3/8]    last reason: 3/7: tensor 'self._modules['conv']._parameters['weight']' size mismatch at index 0. expected 768, actual 96. Guard failed on a parameter, consider using torch._dynamo.config.force_parameter_static_shapes = False to allow dynamism on parameters.
W0827 16:36:06.162000 6470 torch/_dynamo/convert_frame.py:1743] [3/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0827 16:36:06.162000 6470 torch/_dynamo/convert_frame.py:1743] [3/8] To diagnose recompilation issues, see https://docs.pytorch.org/docs/main/user_guide/torch_compiler/compile/programming_model.recompilation.html


CUDA out of memory. Tried to allocate 2.34 GiB. GPU 0 has a total capacity of 39.49 GiB of which 1.27 GiB is free. Including non-PyTorch memory, this process has 38.21 GiB memory in use. Of the allocated memory 37.36 GiB is allocated by PyTorch, and 316.54 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
AutoBatch: Using batch-size 3 for CUDA:0 23.43G/39.49G (59%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2736.6±614.2 MB/s, size: 221.7 KB)
train: Scanning /content/football-players-detection-1/train/labels.cache... 612 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 612/612 197.5Mit/s 0.0s
train: Caching images (1.6GB RAM): 100% ━━━━━━━━━━━━ 612/612 488.6it/s 1.3s
albumentations: Blur(p=0.01, blur_limi

In [ ]:
BEST    = RUN_DIR / "weights" / "best.pt"
model   = YOLO(str(BEST))
metrics = model.val(data=str(DATA), imgsz=IMGSZ, split="val")

for i, c in enumerate(metrics.box.ap_class_index):
    print(f"{model.names[c]:<12} AP50={metrics.box.ap50[i]:.3f}  AP50-95={metrics.box.ap[i]:.3f}")
print(f"\nmAP50={metrics.box.map50:.3f}  mAP50-95={metrics.box.map:.3f}")

Ultralytics 8.4.130 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO26x summary (fused): 190 layers, 55,638,168 parameters, 0 gradients, 193.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1736.7±1556.2 MB/s, size: 232.6 KB)
val: Scanning /content/football-players-detection-1/valid/labels.cache... 38 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 38/38 13.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.0it/s 2.9s
                   all         38        905      0.962      0.885       0.91       0.68
                  ball         35         35      0.957      0.641      0.689      0.352
            goalkeeper         27         27       0.97      0.926      0.982      0.806
                player         38        754      0.976      0.984      0.994       0.84
               referee         38         89      0.946      0.989      0.977      0.723
Speed: 8.0m

In [ ]:
dummy = np.zeros((720, 1280, 3), dtype=np.uint8)

def bench(m, half=False, n=50):
    for _ in range(10):
        m.predict(dummy, imgsz=IMGSZ, device=0, half=half, verbose=False)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(n):
        m.predict(dummy, imgsz=IMGSZ, device=0, half=half, verbose=False)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1000

fp32 = bench(model)
fp16 = bench(model, half=True)
print(f"fp32 {fp32:.1f} ms ({1000/fp32:.1f} FPS) | fp16 {fp16:.1f} ms ({1000/fp16:.1f} FPS)")
print("obs: A100 é ~3-5x a RTX 3060 local; escale o número por aí.")

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.